##Hadoop Installation Part
Hadoop is a Java-based programming framework that supports the processing and storage of extremely large datasets on a cluster of inexpensive machines. It was the first major open source project in the big data playing field and is sponsored by the Apache Software Foundation.

## Step 1:Installing Hadoop

In [ ]:
!wget https://downloads.apache.org/hadoop/common/hadoop-3.4.1/hadoop-3.4.1.tar.gz

In [ ]:
!tar -xzvf hadoop-3.4.1.tar.gz && cp -r hadoop-3.4.1/ /usr/local/

## Step 2:Configuring Hadoop’s Java Home
Hadoop requires that you set the path to Java, either as an environment variable or in the Hadoop configuration file.

In [ ]:
#To find the default Java path
!readlink -f /usr/bin/java | sed "s:bin/java::"

To set java path, go to /usr/local/hadoop-3.3.0/etc/hadoop/hadoop-env.sh then

. . .
export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64/
 . . .

# Step 3:  Running Hadoop

---

### Set up Hadoop's Java ☕

In [ ]:
#To find the default Java path and add export in hadoop-env.sh
JAVA_HOME = !readlink -f /usr/bin/java | sed "s:bin/java::"
java_home_text = JAVA_HOME[0]
java_home_text_command = f"$ {JAVA_HOME[0]} "
!echo export JAVA_HOME=$java_home_text >>/usr/local/hadoop-3.4.1/etc/hadoop/hadoop-env.sh

---

### Set Hadoop home variables 🏡

In [ ]:
import os
os.environ['HADOOP_HOME']="/usr/local/hadoop-3.4.1"
os.environ['JAVA_HOME']=java_home_text

In [ ]:
#Running Hadoop
!/usr/local/hadoop-3.4.1/bin/hadoop

In [ ]:
!mkdir ~/input
!cp /usr/local/hadoop-3.4.1/etc/hadoop/*.xml ~/input

In [ ]:
!ls ~/input

In [ ]:
# !/usr/local/hadoop-3.4.1/bin/hadoop jar /usr/local/hadoop-3.4.1/share/hadoop/mapreduce/hadoop-mapreduce-examples-3.4.1.jar grep ~/input ~/grep_example 'allowed[.]*'

In [ ]:
# !cat ~/grep_example/*

**Download 20newsgroups dataset available at** http://qwone.com/~jason/20Newsgroups.

In [ ]:
!wget http://qwone.com/~jason/20Newsgroups/20news-18828.tar.gz

!tar -xzvf 20news-18828.tar.gz

## Hadoop Streaming
The example below uses `-files` (rather than deprecated `-file`) and checks inputs before running. The two cgroup warnings are runtime notices, not the cause of the missing mapper error.


In [ ]:
!find / -path /proc -prune -o -name 'hadoop-streaming*.jar' -print
# kill the process by control + C if takes too long to find stream

## Add mapper and reducer scripts from Google Drive
The notebook is stored in Google Drive, but Colab runs under `/content`. The next cell mounts Drive, locates the Python scripts, and copies them to `/content` before launching Hadoop Streaming.


In [ ]:
# Mount Google Drive and resolve the Hadoop Streaming scripts.
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')
COURSE_DIR = Path('/content/drive/MyDrive/Colab Notebooks/UCI/UCI_425.07_Big_Data_Analytics/03_In-Class Practice')
CODEBOOK_DIR = COURSE_DIR / 'Codebook'
print('Course directory exists:', COURSE_DIR.is_dir())
print('Codebook directory exists:', CODEBOOK_DIR.is_dir())

# Hadoop executes scripts from /content. Copy from Drive first.
# The original exercise calls mapper.py; mapper2.py is accepted as an alternative.
def locate_script(names):
    for folder in (CODEBOOK_DIR, COURSE_DIR, Path('/content')):
        for name in names:
            candidate = folder / name
            if candidate.is_file():
                return candidate
    return None

mapper_source = locate_script(('mapper.py', 'mapper2.py'))
reducer_source = locate_script(('reducer.py',))
if mapper_source is None or reducer_source is None:
    print('Available Python files in the course folder:')
    if COURSE_DIR.is_dir():
        for p in sorted(COURSE_DIR.rglob('*.py'))[:40]:
            print(' ', p)
    raise FileNotFoundError(
        'Missing mapper.py (or mapper2.py) and/or reducer.py. '
        'Place the scripts in the Google Drive Codebook folder, or upload them to /content.'
    )

MAPPER = Path('/content/mapper.py')
REDUCER = Path('/content/reducer.py')
if mapper_source != MAPPER:
    shutil.copy2(mapper_source, MAPPER)
if reducer_source != REDUCER:
    shutil.copy2(reducer_source, REDUCER)
print('Mapper source:', mapper_source)
print('Reducer source:', reducer_source)
print('Ready:', MAPPER.is_file(), REDUCER.is_file())


In [ ]:
# Clean only the output folder for this example.
from pathlib import Path
import shutil
OUTPUT = Path('/content/output')
if OUTPUT.exists():
    shutil.rmtree(OUTPUT)
print('Output directory ready:', OUTPUT)


In [ ]:
# Hadoop Streaming: use -files instead of deprecated -file.
# The input is the same 20 Newsgroups document used in the original lesson.
import subprocess
from pathlib import Path

INPUT = Path('/content/20news-18828/alt.atheism/49960')
STREAMING_JAR = Path('/usr/local/hadoop-3.4.1/share/hadoop/tools/lib/hadoop-streaming-3.4.1.jar')
HADOOP = '/usr/local/hadoop-3.4.1/bin/hadoop'
for required in (INPUT, STREAMING_JAR, MAPPER, REDUCER):
    if not required.is_file():
        raise FileNotFoundError(f'Required input missing: {required}')

command = [
    HADOOP, 'jar', str(STREAMING_JAR),
    '-D', 'mapreduce.framework.name=local',
    '-files', f'{MAPPER},{REDUCER}',
    '-input', str(INPUT),
    '-output', str(OUTPUT),
    '-mapper', 'python3 mapper.py',
    '-reducer', 'python3 reducer.py',
]
print('Running Hadoop Streaming with:', INPUT)
subprocess.run(command, check=True)
print('Job completed. Output:', OUTPUT)


In [ ]:
# Inspect output files after a successful job.
from pathlib import Path
if not OUTPUT.is_dir():
    raise FileNotFoundError('Hadoop output missing: run the streaming job above first.')
print('\n'.join(p.name for p in sorted(OUTPUT.iterdir())))


In [ ]:
# Display Hadoop Streaming output.
parts = sorted(OUTPUT.glob('part-*'))
if not parts:
    raise FileNotFoundError('No part-* output found; check the Hadoop job log above.')
for part in parts:
    print(part.read_text(errors='replace')[:12000])
